In [1]:
import os
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score
import wandb

In [2]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.RandomCrop(size=(128, 256))


class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
        spect_dbs = [
            torch.tensor(
                librosa.power_to_db(spec, ref=np.max),
                dtype=torch.float32
            )
            for spec in spectrograms
        ]

        spect_dbs = torch.stack(spect_dbs)

        mean = spect_dbs.mean(dim=0)
        std = spect_dbs.std(dim=0)

        self.spect_dbs = (spect_dbs - mean) / std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return transform(self.spect_dbs[idx]), self.labels[idx]

In [5]:
os.listdir('/kaggle/input/dev_data/dev_data/slider/train')

['normal_id_02_00000546.wav',
 'normal_id_02_00000764.wav',
 'normal_id_00_00000530.wav',
 'normal_id_02_00000820.wav',
 'normal_id_02_00000102.wav',
 'normal_id_02_00000838.wav',
 'normal_id_02_00000270.wav',
 'normal_id_00_00000185.wav',
 'normal_id_02_00000796.wav',
 'normal_id_00_00000723.wav',
 'normal_id_02_00000895.wav',
 'normal_id_00_00000698.wav',
 'normal_id_02_00000394.wav',
 'normal_id_02_00000674.wav',
 'normal_id_02_00000440.wav',
 'normal_id_02_00000966.wav',
 'normal_id_02_00000045.wav',
 'normal_id_04_00000372.wav',
 'normal_id_02_00000903.wav',
 'normal_id_04_00000207.wav',
 'normal_id_02_00000092.wav',
 'normal_id_00_00000843.wav',
 'normal_id_00_00000725.wav',
 'normal_id_02_00000380.wav',
 'normal_id_02_00000616.wav',
 'normal_id_04_00000198.wav',
 'normal_id_02_00000856.wav',
 'normal_id_00_00000491.wav',
 'normal_id_00_00000106.wav',
 'normal_id_00_00000219.wav',
 'normal_id_02_00000461.wav',
 'normal_id_02_00000519.wav',
 'normal_id_04_00000237.wav',
 'normal_i

In [6]:
train_dataset = AudioDataset('/kaggle/input/dev_data/dev_data/slider/train', train=True)
test_dataset = AudioDataset('/kaggle/input/dev_data/dev_data/slider/test', train=False)

In [7]:
batch_size = 16

train_set, validation_set = torch.utils.data.random_split(train_dataset, [0.8, 0.2], generator=generator)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)
validation_loader = torch.utils.data.DataLoader(
    validation_set,
    batch_size=batch_size,
    shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [8]:
class CNNAE(nn.Module):
    def __init__(self):
        super(CNNAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),   # (32, 128, 256)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (32, 64, 128)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # (64, 64, 128)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (64, 32, 64)
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # (128, 32, 64)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (128, 16, 32)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),   # (64, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),    # (32, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),     # (1, 128, 256)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.squeeze(1)


class C1DNNAE(nn.Module):
    def __init__(self):
        super(C1DNNAE, self).__init__()
        # Encoder: input shape (batch, 128, 256)
        self.encoder = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),   # (batch, 64, 128)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),    # (batch, 32, 64)
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=5, stride=2, padding=2),    # (batch, 16, 32)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(16, 32, kernel_size=4, stride=2, padding=1),  # (batch, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),  # (batch, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 256)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = self.encoder(x)
        x = self.decoder(x)
        return x


class C1DNNAE_INV(nn.Module):
    def __init__(self):
        super(C1DNNAE_INV, self).__init__()
        # Encoder: input shape (batch, 256, 128)
        self.encoder = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=5, stride=2, padding=2),   # (batch, 128, 64)
            nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),    # (batch, 64, 32)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),     # (batch, 32, 16)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),   # (batch, 64, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(128, 256, kernel_size=4, stride=2, padding=1),  # (batch, 256, 128)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = x.permute(0, 2, 1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.permute(0, 2, 1)


class LAE(nn.Module):
    def __init__(self):
        super(LAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(128 * 256, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, 128 * 256),
            nn.Tanh()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.view(x.size(0), 128, 256)

In [9]:
def train_one_epoch(model, data_loader, optimizer, criterion, scheduler):
    model.train()

    total_loss = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        # print(inputs.shape)
        # print(outputs.shape)

        loss = criterion(outputs, inputs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    return total_loss / len(data_loader)


def validate(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            total_loss += loss.item()

    return total_loss / len(data_loader)


def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).reshape(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors


def test(model, data_loader):
    model.eval()

    errors = compute_reconstruction_errors(model, data_loader)

    roc_auc_scores = roc_auc_score(test_dataset.labels.cpu(), errors)
    print(f"ROC AUC Score test: {roc_auc_scores}")

    return roc_auc_scores

In [10]:
def train(lr, step_size, gamma, epochs, use_wandb=False):
    model = C1DNNAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss = validate(model, validation_loader, criterion)
        roc_auc_score = test(model, test_loader)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        if use_wandb:
            wandb.log({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "roc_auc_score": roc_auc_score,
                "epoch": epoch + 1,
            })

    return model

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret('wandb-api-key')

wandb.login(key=key)

sweep_configuration = {
    "method": "grid",
    "metric": {"goal": "maximize", "name": "roc_auc_score"},
    'name': "convolutional_1d_autoencoder",
    "parameters": {
        "lr": {'values': [1e-2, 1e-3, 1e-4]},
        "step_size": {'values': [3, 5, 7, 10]},
        "gamma": {'values': [0.01, 0.1, 0.25, 0.5]},
    },
}


def train_wrapper():
    with wandb.init() as run:
        train(
            lr=run.config.lr,
            step_size=run.config.step_size,
            gamma=run.config.gamma,
            epochs=50,
            use_wandb=True
        )


sweep_id = wandb.sweep(sweep=sweep_configuration, entity='matteo-ghia-politecnico-di-torino', project="aml challenge 2")
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=train_wrapper)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: francescogiannuzzo2002-fg (miriam-lamari2-eurecom) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Agent Starting Run: 85zdjxbp with config:
wandb: 	gamma: 0.01
wandb: 	lr: 0.01
wandb: 	step_size: 10
wandb: Currently logged in as: francescogiannuzzo2002-fg (matteo-ghia-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle/working/wandb/run-20250523_121844-85zdjxbp
w

ROC AUC Score test: 0.5890511860174782
Epoch 1/20, Train Loss: 0.6210, Validation Loss: 0.4736
ROC AUC Score test: 0.53173949230129
Epoch 2/20, Train Loss: 0.4759, Validation Loss: 0.4731
ROC AUC Score test: 0.5711943404078236
Epoch 3/20, Train Loss: 0.4634, Validation Loss: 0.4598
ROC AUC Score test: 0.5968580940491053
Epoch 4/20, Train Loss: 0.4458, Validation Loss: 0.4232
ROC AUC Score test: 0.5882979608822305
Epoch 5/20, Train Loss: 0.4427, Validation Loss: 0.4132
ROC AUC Score test: 0.6098543487307533
Epoch 6/20, Train Loss: 0.4432, Validation Loss: 0.4093
ROC AUC Score test: 0.621485642946317
Epoch 7/20, Train Loss: 0.4457, Validation Loss: 0.4383
ROC AUC Score test: 0.612929671244278
Epoch 8/20, Train Loss: 0.4154, Validation Loss: 0.3923
ROC AUC Score test: 0.6240491052850603
Epoch 9/20, Train Loss: 0.4091, Validation Loss: 0.3902
ROC AUC Score test: 0.5984186433624636
Epoch 10/20, Train Loss: 0.4246, Validation Loss: 0.4023
ROC AUC Score test: 0.6055680399500625
Epoch 11/20, T

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb: roc_auc_score ▅▁▄▆▅▇█▇█▆▇▇▇▇▇▇▇██▇
wandb:    train_loss █▄▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss ██▇▄▄▃▅▂▂▃▂▂▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 20
wandb: roc_auc_score 0.61695
wandb:    train_loss 0.3871
wandb:      val_loss 0.37901
wandb: 
wandb: 🚀 View run earthy-sweep-4 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/85zdjxbp
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_121844-85zdjxbp/logs
wandb: Agent Starting Run: rxv74jg3 with config:
wandb: 	gamma: 0.01
wandb: 	lr: 0.001
wandb: 	step_size: 3
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle/worki

ROC AUC Score test: 0.5059300873907615
Epoch 1/20, Train Loss: 0.6204, Validation Loss: 0.4938
ROC AUC Score test: 0.5967956720765709
Epoch 2/20, Train Loss: 0.4315, Validation Loss: 0.3932
ROC AUC Score test: 0.6004494382022473
Epoch 3/20, Train Loss: 0.3967, Validation Loss: 0.3754
ROC AUC Score test: 0.6016271327507283
Epoch 4/20, Train Loss: 0.3853, Validation Loss: 0.3721
ROC AUC Score test: 0.5990178942987932
Epoch 5/20, Train Loss: 0.3839, Validation Loss: 0.3712
ROC AUC Score test: 0.5991136079900126
Epoch 6/20, Train Loss: 0.3830, Validation Loss: 0.3706
ROC AUC Score test: 0.599371618809821
Epoch 7/20, Train Loss: 0.3829, Validation Loss: 0.3709
ROC AUC Score test: 0.6007324178110696
Epoch 8/20, Train Loss: 0.3832, Validation Loss: 0.3710
ROC AUC Score test: 0.6002496878901373
Epoch 9/20, Train Loss: 0.3825, Validation Loss: 0.3710
ROC AUC Score test: 0.5997045359966708
Epoch 10/20, Train Loss: 0.3830, Validation Loss: 0.3707
ROC AUC Score test: 0.5994673325010403
Epoch 11/20

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb: roc_auc_score ▁███████████████████
wandb:    train_loss █▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 20
wandb: roc_auc_score 0.60026
wandb:    train_loss 0.38276
wandb:      val_loss 0.37042
wandb: 
wandb: 🚀 View run crisp-sweep-5 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/rxv74jg3
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_121905-rxv74jg3/logs
wandb: Agent Starting Run: hc8okfgd with config:
wandb: 	gamma: 0.01
wandb: 	lr: 0.001
wandb: 	step_size: 5
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle/worki

...

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb: roc_auc_score ▁▃▃▂▆▇▇█████████████
wandb:    train_loss █▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▅▅▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 20
wandb: roc_auc_score 0.58272
wandb:    train_loss 0.40154
wandb:      val_loss 0.3882
wandb: 
wandb: 🚀 View run driven-sweep-34 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/v3iwqu89
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_122916-v3iwqu89/logs
wandb: Agent Starting Run: dpgfp71u with config:
wandb: 	gamma: 0.5
wandb: 	lr: 0.0001
wandb: 	step_size: 7
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle/work

ROC AUC Score test: 0.5652143154390346
Epoch 1/20, Train Loss: 0.9183, Validation Loss: 0.6567
ROC AUC Score test: 0.5077361631294215
Epoch 2/20, Train Loss: 0.5522, Validation Loss: 0.5208
ROC AUC Score test: 0.4942571785268415
Epoch 3/20, Train Loss: 0.5215, Validation Loss: 0.5095
ROC AUC Score test: 0.5067790262172285
Epoch 4/20, Train Loss: 0.4962, Validation Loss: 0.4544
ROC AUC Score test: 0.5762130669995839
Epoch 5/20, Train Loss: 0.4344, Validation Loss: 0.4040
ROC AUC Score test: 0.5800041614648356
Epoch 6/20, Train Loss: 0.4146, Validation Loss: 0.3984
ROC AUC Score test: 0.580923845193508
Epoch 7/20, Train Loss: 0.4109, Validation Loss: 0.3946
ROC AUC Score test: 0.5841323345817727
Epoch 8/20, Train Loss: 0.4074, Validation Loss: 0.3935
ROC AUC Score test: 0.5831543903454016
Epoch 9/20, Train Loss: 0.4064, Validation Loss: 0.3913
ROC AUC Score test: 0.582501040366209
Epoch 10/20, Train Loss: 0.4053, Validation Loss: 0.3906
ROC AUC Score test: 0.5848439450686642
Epoch 11/20,

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb: roc_auc_score ▆▂▁▂▇▇▇█████████████
wandb:    train_loss █▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 20
wandb: roc_auc_score 0.58866
wandb:    train_loss 0.38713
wandb:      val_loss 0.37304
wandb: 
wandb: 🚀 View run chocolate-sweep-35 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/dpgfp71u
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_122937-dpgfp71u/logs
wandb: Agent Starting Run: nquvv2n7 with config:
wandb: 	gamma: 0.5
wandb: 	lr: 0.0001
wandb: 	step_size: 10
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle

ROC AUC Score test: 0.6030045776113192
Epoch 1/20, Train Loss: 0.9046, Validation Loss: 0.7451
ROC AUC Score test: 0.515318352059925
Epoch 2/20, Train Loss: 0.5782, Validation Loss: 0.5243
ROC AUC Score test: 0.5039409071993342
Epoch 3/20, Train Loss: 0.5259, Validation Loss: 0.5144
ROC AUC Score test: 0.4971244277985851
Epoch 4/20, Train Loss: 0.5191, Validation Loss: 0.5102
ROC AUC Score test: 0.5038743237619643
Epoch 5/20, Train Loss: 0.5159, Validation Loss: 0.5071
ROC AUC Score test: 0.5001706200582605
Epoch 6/20, Train Loss: 0.5115, Validation Loss: 0.4985
ROC AUC Score test: 0.5763712026633375
Epoch 7/20, Train Loss: 0.4614, Validation Loss: 0.4067
ROC AUC Score test: 0.5789346650020808
Epoch 8/20, Train Loss: 0.4127, Validation Loss: 0.3952
ROC AUC Score test: 0.5796712442779859
Epoch 9/20, Train Loss: 0.4064, Validation Loss: 0.3914
ROC AUC Score test: 0.5824594257178527
Epoch 10/20, Train Loss: 0.4038, Validation Loss: 0.3890
ROC AUC Score test: 0.5831876820640866
Epoch 11/20

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb: roc_auc_score █▂▁▁▁▁▆▆▆▇▇▇▇▇▇▇▇▇▇▇
wandb:    train_loss █▄▃▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▄▄▄▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 20
wandb: roc_auc_score 0.59334
wandb:    train_loss 0.38356
wandb:      val_loss 0.36923
wandb: 
wandb: 🚀 View run whole-sweep-36 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/nquvv2n7
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_122956-nquvv2n7/logs
wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


In [12]:
# model = train(lr=1e-3, weight_decay=1e-8, epochs=50)
# torch.save(model.state_dict(), "model.pt")